# CNN — Identify Fruit from Image
**Dataset:** Fake colored 8×8 images (no download needed)
**Input:** RGB image | **Output:** Apple / Banana / Orange

> CNN = Convolutional Neural Network. Filters scan the image to detect color/shape patterns.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

np.random.seed(42)

# ── Create Fake Fruit Images ──────────────────────────────
# Each image: 8x8 pixels, 3 color channels (RGB)
# Apple  = mostly RED,    Banana = mostly YELLOW,  Orange = RED + some GREEN
def make_images(label, n=40):
    imgs = []
    for _ in range(n):
        img = np.random.uniform(0, 0.1, (8, 8, 3))
        if label == 0:    # Apple
            img[:,:,0] = np.random.uniform(0.8, 1.0, (8,8))
        elif label == 1:  # Banana
            img[:,:,0] = np.random.uniform(0.8, 1.0, (8,8))
            img[:,:,1] = np.random.uniform(0.8, 1.0, (8,8))
        else:             # Orange
            img[:,:,0] = np.random.uniform(0.8, 1.0, (8,8))
            img[:,:,1] = np.random.uniform(0.4, 0.6, (8,8))
        imgs.append(img)
    return np.array(imgs)

X = np.vstack([make_images(0), make_images(1), make_images(2)])
y = np.array([0]*40 + [1]*40 + [2]*40)

# ── Train / Test Split ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Images shape: {X.shape} | Train: {len(X_train)} | Test: {len(X_test)}')

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# ── Show Sample Fruit Images ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for i, (idx, name) in enumerate([(0,'Apple'), (40,'Banana'), (80,'Orange')]):
    axes[i].imshow(X[idx])
    axes[i].set_title(name); axes[i].axis('off')
plt.suptitle('Sample Fruit Images (Synthetic Colors)')
plt.show()

In [ ]:
# ── Build CNN ─────────────────────────────────────────────
model = Sequential([
    Input(shape=(8, 8, 3)),            # 8x8 RGB image
    Conv2D(16, (3,3), activation='relu'),
    MaxPooling2D((2,2)),               # reduce spatial size
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')    # 3 fruits
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────
history = model.fit(X_train, y_train, epochs=30, batch_size=16,
                    validation_data=(X_test, y_test), verbose=0)

print(f'Train Accuracy: {history.history["accuracy"][-1]*100:.1f}%')
print(f'Test  Accuracy: {history.history["val_accuracy"][-1]*100:.1f}%')

In [ ]:
# ── Accuracy Plot ─────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train')
plt.plot(history.history['val_accuracy'], label='Test', linestyle='--')
plt.title('CNN — Training Accuracy over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                              display_labels=['Apple','Banana','Orange'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — CNN Fruit Classifier')
plt.show()

In [ ]:
# ── Predict on Sample Test Images ────────────────────────
fruit_names = ['Apple', 'Banana', 'Orange']
fig, axes = plt.subplots(1, 3, figsize=(9, 3))

for i, idx in enumerate([0, 40, 80]):
    pred = model.predict(X[idx:idx+1], verbose=0).argmax()
    axes[i].imshow(X[idx])
    axes[i].set_title(f'True: {fruit_names[y[idx]]}\nPred: {fruit_names[pred]}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('CNN Fruit Image Predictions')
plt.show()